# Week 3 Summary: Serving Layer

Days 15–19. FastAPI serving layer, ONNX export, latency profiling, batch endpoint, Docker.

## What Was Built

| Day | Deliverable |
|-----|-------------|
| 15 | FastAPI /classify endpoint, DuckDB request logging |
| 16 | ONNX int8 export, PyTorch inference replaced with ORT |
| 17 | Latency profiling, Risk Gate 2 passed (p99 = 91ms) |
| 18 | /classify-batch with true batching, 17/17 tests |
| 19 | Dockerfile, docker-compose.yml, one-command local stack |

## Latency Story

| Backend | Single request (warm) | p99 (1 user, 60s) |
|---|---|---|
| PyTorch CPU (Day 15) | 116ms | not measured |
| ONNX int8 (Day 16+) | 66ms | 91ms |

ONNX int8 quantization reduced single-request latency by 43%.
Risk Gate 2 criterion: p99 < 100ms. Result: **PASSED**.

## Key Bugs Fixed

1. **DuckDB PRIMARY KEY contention**: concurrent requests computed identical MAX(id)+1, causing 28% failure rate at 50 RPS. Fixed by removing PRIMARY KEY constraint.
2. **Blocking inference in async handler**: ONNX inference inside async def blocked the event loop, causing p99 = 3300ms at 50 users. Fixed by offloading to run_in_executor.
3. **asyncio.gather for batch inference**: 32 concurrent thread pool jobs gave no speedup on CPU. Fixed by true batching: one tokenizer call, one ONNX forward pass.

## Architecture at End of Week 3

```
POST /classify        -> tokenize -> ONNX forward pass -> softmax -> log (async)
POST /classify-batch  -> batch tokenize -> ONNX forward pass -> split -> log (async)
GET  /health          -> {status: ok}
```

All requests served via ONNX Runtime int8 model from checkpoints/v1-onnx-int8/.
Request logs written to DuckDB request_log table asynchronously.
